Databricks notebook source
MAGIC %md
MAGIC # Silver: clean and standardise clinical trial data
MAGIC
MAGIC **Flow:** new Bronze rows → standardise → de-duplicate (latest `modified_date` wins) → validate →
MAGIC quarantine failures → MERGE the rest into `silver.clinical_trial`.
MAGIC
MAGIC * Only Bronze rows loaded since the last successful run are processed (`silver.etl_control` holds the watermark).
MAGIC * Re-running is safe: MERGE and the quarantine write are idempotent, and the watermark moves last.
MAGIC * The job stops (and the Silver MERGE is skipped) when too many rows fail validation.

In [0]:
# COMMAND ----------
import os, sys, uuid, datetime as dt

In [0]:
# make the repo's src/ folder importable (this notebook lives in src/silver)
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

In [0]:
from common.silver_transforms import standardise, dedup_latest, to_silver
from common.silver_rules import RULES
from common.dq_checks import apply_rules, split_clean_quarantine, summarise, DQLog


In [0]:
# COMMAND ----------
dbutils.widgets.text("catalog", "clinical_dev")
dbutils.widgets.text("max_quarantine_pct", "10")     # fail the run if more than this % of rows are quarantined
dbutils.widgets.dropdown("full_refresh", "false", ["false", "true"])

In [0]:
catalog = dbutils.widgets.get("catalog")
max_quarantine_pct = float(dbutils.widgets.get("max_quarantine_pct"))
full_refresh = dbutils.widgets.get("full_refresh") == "true"

In [0]:
PIPELINE = "silver_clinical_trial"
bronze_tbl = f"{catalog}.bronze.clinical_trial_raw"
silver_tbl = f"{catalog}.silver.clinical_trial"
quar_tbl = f"{catalog}.silver.clinical_trial_quarantine"
results_tbl = f"{catalog}.silver.dq_results"
control_tbl = f"{catalog}.silver.etl_control"
run_id = str(uuid.uuid4())

COMMAND ----------
MAGIC %md ## 1. Create the Silver tables (first run only)

In [0]:
# COMMAND ----------
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {silver_tbl} (
  subject_id STRING NOT NULL, trial_id STRING, visit_name STRING NOT NULL, visit_date DATE NOT NULL,
  site_id STRING, site_name STRING, country STRING, investigator_name STRING,
  age INT, gender STRING, treatment_arm STRING, study_status STRING,
  weight_kg DOUBLE, height_cm DOUBLE, bmi DOUBLE,
  systolic_bp INT, diastolic_bp INT, heart_rate INT,
  hba1c_pct DOUBLE, fasting_glucose_mg_dl INT, adherence_pct DOUBLE,
  ae_flag BOOLEAN, ae_term STRING, ae_severity STRING, serious_ae BOOLEAN, remarks STRING,
  source_record_id INT, created_date TIMESTAMP, modified_date TIMESTAMP,
  dq_fixes ARRAY<STRING>, dq_warnings ARRAY<STRING>,
  _bronze_ingest_time TIMESTAMP, _source_file STRING, _silver_loaded_at TIMESTAMP
) COMMENT 'One row per subject and visit, cleaned and validated'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {quar_tbl} (
  record_id INT, subject_id STRING, visit_name STRING, modified_date TIMESTAMP,
  dq_errors ARRAY<STRING>, dq_warnings ARRAY<STRING>, raw_record STRING,
  _ingest_time TIMESTAMP, _source_file STRING, quarantined_at TIMESTAMP, run_id STRING
) COMMENT 'Rows that failed an ERROR rule. raw_record holds the untouched source row'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {results_tbl} (
  run_id STRING, layer STRING, check_name STRING, severity STRING, status STRING,
  metric DOUBLE, threshold STRING, details STRING, checked_at TIMESTAMP
)""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {control_tbl} (
  pipeline STRING, last_ingest_time TIMESTAMP, updated_at TIMESTAMP, run_id STRING
)""")

In [0]:
# Delta CHECK constraints are a last line of defence: a write that breaks them fails outright
for name, expr in [
    ("valid_hba1c", "hba1c_pct IS NULL OR hba1c_pct BETWEEN 4 AND 15"),
    ("valid_heart_rate", "heart_rate IS NULL OR heart_rate BETWEEN 30 AND 220"),
    ("valid_bp_order", "systolic_bp IS NULL OR diastolic_bp IS NULL OR systolic_bp > diastolic_bp"),
]:
    try:
        spark.sql(f"ALTER TABLE {silver_tbl} ADD CONSTRAINT {name} CHECK ({expr})")
    except Exception as e:
        if "already exists" not in str(e).lower():
            raise

COMMAND ----------
MAGIC %md ## 2. Read only the new Bronze rows

In [0]:
# COMMAND ----------
if full_refresh:
    spark.sql(f"DELETE FROM {control_tbl} WHERE pipeline = '{PIPELINE}'")
    spark.sql(f"TRUNCATE TABLE {quar_tbl}")

In [0]:
last_wm = spark.table(control_tbl).where(F.col("pipeline") == PIPELINE).agg(F.max("last_ingest_time")).first()[0]
last_wm = last_wm or dt.datetime(1900, 1, 1)

In [0]:
new_rows = spark.table(bronze_tbl).where(F.col("_ingest_time") > F.lit(last_wm))
n_in = new_rows.count()
print(f"run_id={run_id}  watermark={last_wm}  new Bronze rows={n_in}")
if n_in == 0:
    dbutils.notebook.exit("No new Bronze rows to process")
new_wm = new_rows.agg(F.max("_ingest_time")).first()[0]

COMMAND ----------
MAGIC %md ## 3. Standardise, de-duplicate and validate

In [0]:
# COMMAND ----------
# subject-level facts missing in this batch (age, gender, height, arm) are filled from Silver
reference = (spark.table(silver_tbl).groupBy("subject_id").agg(
    F.max("age").alias("ref_age"),
    F.max(F.when(F.col("gender").isin("M", "F"), F.col("gender"))).alias("ref_gender"),
    F.max("height_cm").alias("ref_height_cm"),
    F.max("treatment_arm").alias("ref_treatment_arm")))

In [0]:
standardised = standardise(new_rows, reference)
deduped = dedup_latest(standardised)          # the newest version of each subject + visit wins
checked = apply_rules(deduped, RULES)         # adds dq_errors and dq_warnings
clean, quarantine = split_clean_quarantine(checked)

In [0]:
stats = summarise(checked, RULES)
n_deduped = stats["total"]
n_superseded = n_in - n_deduped
quarantine_pct = 100.0 * stats["error_rows"] / max(n_deduped, 1)

COMMAND ----------
MAGIC %md ## 4. Log the results

In [0]:
# COMMAND ----------
log = DQLog(run_id, "silver")
log.info("rows_in", n_in)
log.info("superseded_duplicates", n_superseded, "older versions of a subject + visit, replaced by a newer row")
log.info("rows_after_dedup", n_deduped)
log.info("rows_quarantined", stats["error_rows"])
log.info("rows_with_warnings", stats["warn_rows"])
for rule, severity, n_failed in stats["by_rule"]:
    # QUARANTINE / FLAG rows describe what a rule did to the data. They never fail the job.
    log.add(rule, "QUARANTINE" if severity == "ERROR" else "FLAG", n_failed == 0, n_failed, "0 rows",
            f"{n_failed} of {n_deduped} rows" if n_failed else "")
for row in checked.select(F.explode("dq_fixes").alias("fix")).groupBy("fix").count().collect():
    log.info(f"fix:{row['fix']}", row["count"])
# this is the run-level gate
log.add("quarantine_rate_pct", "ERROR", quarantine_pct <= max_quarantine_pct, quarantine_pct,
        f"<= {max_quarantine_pct}%")

In [0]:
results = log.to_df(spark)
results.write.mode("append").saveAsTable(results_tbl)
display(results.orderBy(F.col("status").desc(), "severity", "check_name"))

COMMAND ----------
MAGIC %md ## 5. Write quarantine (always), then stop if the batch is clearly broken

In [0]:
# COMMAND ----------
quarantine_out = (quarantine.select("record_id", "subject_id", "visit_name", "modified_date",
                                    "dq_errors", "dq_warnings", "raw_record", "_ingest_time", "_source_file")
                  .withColumn("quarantined_at", F.current_timestamp())
                  .withColumn("run_id", F.lit(run_id)))

In [0]:
(DeltaTable.forName(spark, quar_tbl).alias("t")
    .merge(quarantine_out.alias("s"), "t.record_id = s.record_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

In [0]:
if log.failed_errors():
    raise Exception(f"Silver stopped before the MERGE. Failed run-level checks: {log.failed_errors()}. "
                    f"Quarantine rate was {quarantine_pct:.1f}% (limit {max_quarantine_pct}%). "
                    f"The watermark was not moved, so the next run reprocesses these rows.")

COMMAND ----------
MAGIC %md ## 6. MERGE into Silver

In [0]:
# COMMAND ----------
silver_rows = to_silver(clean)

In [0]:
(DeltaTable.forName(spark, silver_tbl).alias("t")
    .merge(silver_rows.alias("s"), "t.subject_id = s.subject_id AND t.visit_name = s.visit_name")
    .whenMatchedUpdateAll(condition="s.modified_date >= t.modified_date")   # never let an older version overwrite a newer one
    .whenNotMatchedInsertAll()
    .execute())

In [0]:
# a quarantined row is resolved once Silver holds the same subject + visit with a newer or equal modified_date
(DeltaTable.forName(spark, quar_tbl).alias("q")
    .merge(spark.table(silver_tbl).select("subject_id", "visit_name", "modified_date").alias("s"),
           "q.subject_id = s.subject_id AND q.visit_name = s.visit_name AND s.modified_date >= q.modified_date")
    .whenMatchedDelete()
    .execute())

COMMAND ----------
MAGIC %md ## 7. Move the watermark (last step, so a failed run is simply retried)

In [0]:
# COMMAND ----------
update = (spark.createDataFrame([(PIPELINE, new_wm, run_id)], "pipeline string, last_ingest_time timestamp, run_id string")
          .withColumn("updated_at", F.current_timestamp()))

In [0]:
(DeltaTable.forName(spark, control_tbl).alias("t")
    .merge(update.alias("s"), "t.pipeline = s.pipeline")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

In [0]:
print(f"Done. in={n_in}  superseded={n_superseded}  quarantined={stats['error_rows']}  "
      f"silver_candidates={n_deduped - stats['error_rows']}  new watermark={new_wm}")